<a href="https://colab.research.google.com/github/annasvenbro/etudesnordiques/blob/main/sudoc_collector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import des paquets nécessaires

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import time  # <--- pour mesurer la durée

# Spécification des chemins et paramètre généraux

In [ ]:
OUTPUT_DIR = Path("data/sru_batches")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 500
RCR_CSV = Path("data/rcr_list.csv")  # chemin vers le CSV contenant la colonne RCR

# Lire la liste des RCR depuis le CSV

In [ ]:
df_rcr = pd.read_csv(RCR_CSV, dtype=str)  # s'assurer que ce sont des chaînes
RCR_LIST = df_rcr["RCR"].dropna().tolist()

# Fonction d'extraction des contributeurs

In [ ]:
def extract_contributors(record, tags, role_code, idref_col, name_col):
    """Extrait les contributeurs (auteurs ou traducteurs) selon le tag et le code $4"""
    idrefs, names = [], []
    for tag in tags:
        for f in record.findall(f".//datafield[@tag='{tag}']"):
            sub4 = f.findtext("subfield[@code='4']")
            if sub4 != role_code:
                continue
            # idRef
            idref = f.findtext("subfield[@code='3']")
            if idref:
                idrefs.append(idref)
            # nom complet
            nom = f.findtext("subfield[@code='a']") or ""
            prenom = f.findtext("subfield[@code='b']") or ""
            annee = f.findtext("subfield[@code='f']") or ""
            full_name = f"{nom}, {prenom} ({annee})" if nom else None
            if full_name:
                names.append(full_name)
    return idrefs, names

# Récupération de toutes les notices pour un RCR donné avec métadonnées enrichies

In [ ]:
def fetch_records(rcr: str):
    records = []
    start = 1

    while True:
        url = (
            f"https://www.sudoc.abes.fr/cbs/sru/"
            f"?operation=searchRetrieve"
            f"&version=1.1"
            f"&recordSchema=unimarc"
            f"&maximumRecords={BATCH_SIZE}"
            f"&startRecord={start}"
            f"&query=rbc%3D{rcr}%20and%20lan%3D%22fre%22%20and%20apu%3E%3D1900%20and%20apu%3C%3D1950"
        )

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        try:
            root = ET.fromstring(response.content)
        except ET.ParseError:
            print(f"⚠️ Erreur XML pour RCR {rcr}, start={start}")
            break

        batch_records = []
        for record in root.findall(".//{*}record"):

            # Filtrer 101$a/101$c
            keep = False
            for f in record.findall(".//datafield[@tag='101']"):
                if f.findtext("subfield[@code='a']") == "fre" and f.findtext("subfield[@code='c']") in ["nor","nob","nno"]:
                    keep = True
                    break
            if not keep:
                continue

            ppn = record.findtext(".//controlfield[@tag='003']")

            # 200 : Titre et responsabilités
            field_200 = record.find(".//datafield[@tag='200']")
            titre = field_200.findtext("subfield[@code='a']") if field_200 is not None else None
            resp_principale = field_200.findtext("subfield[@code='f']") if field_200 is not None else None
            resp_secondaire = field_200.findtext("subfield[@code='g']") if field_200 is not None else None

            # 210 / 214 : Editeur et Date
            publisher, pubdate = None, None
            field_210 = record.find(".//datafield[@tag='210']")
            if field_210 is not None:
                publisher = field_210.findtext("subfield[@code='c']")
                pubdate = field_210.findtext("subfield[@code='d']")

            # Si rien trouvé dans 210, on tente 214
            if not publisher or not pubdate:
                field_214 = record.find(".//datafield[@tag='214']")
                if field_214 is not None:
                        publisher = field_214.findtext("subfield[@code='c']")
                        pubdate = field_214.findtext("subfield[@code='d']")

            # 454 : Titre original
            field_454 = record.find(".//datafield[@tag='454']")
            titre_original = field_454.findtext("subfield[@code='t']") if field_454 is not None else None

            # 700 -> Auteurs ($4=070)
            idref_auteurs, auteurs = extract_contributors(record, ["700"], "070", "IdRef Auteurs", "Auteurs")
            # 701 / 702 -> Traducteurs ($4=730)
            idref_trads, trads = extract_contributors(record, ["701","702"], "730", "IdRef Traducteurs", "Traducteurs")

            batch_records.append({
                "RCR": rcr,
                "PPN": ppn,
                "Titre": titre,
                "Responsabilité principale": resp_principale,
                "Responsabilité secondaire": resp_secondaire,
                "Éditeur": publisher,
                "Date": pubdate,
                "Titre original": titre_original,
                "IdRef Auteurs": idref_auteurs,
                "Auteurs": auteurs,
                "IdRef Traducteurs": idref_trads,
                "Traducteurs": trads
            })

        if not batch_records:
            break

        records.extend(batch_records)

         # Vérifier si on a atteint le total
        number_of_records = root.findtext(".//{*}numberOfRecords")
        if number_of_records is None or start + BATCH_SIZE > int(number_of_records):
            break

        start += BATCH_SIZE

    return records

# Lancement de la récupération

In [ ]:
def main():
    for rcr in tqdm(RCR_LIST, desc="Test SRU"):
        start_time = time.time()
        recs = fetch_records(rcr)
        duration = time.time() - start_time

        if recs:
            df = pd.DataFrame(recs)
            out_file = OUTPUT_DIR / f"sudoc_test_{rcr}.parquet"
            df.to_parquet(out_file, index=False)
            print(f"✅ {len(df)} notices sauvegardées pour RCR {rcr} → {out_file} (temps écoulé : {duration:.2f}s)")
        else:
            print(f"❌ Aucune notice retenue pour RCR {rcr} (temps écoulé : {duration:.2f}s)")

if __name__ == "__main__":
    main()